# BirdNET v2.4 — Post-Training Quantization (PTQ)

## Attempts and findings

**Dynamic range** (cell b1000005, `birdnet_v2.4_int8.tflite`): weights INT8, activations float32 at runtime. 14.2 MB, 72.5% reduction. Predictions wrong (Madagascar Scops-Owl instead of Spotted Antbird).

**Calibrated full INT8** (cell b1000005, `birdnet_v2.4_int8_calibrated.tflite`): weights and activations INT8, 100-clip calibration. 14.1 MB, 72.8% reduction. 0/500 top-1 agreement (cell f0617512).

**Why both fail:** The official Zenodo INT8 model was not built with PTQ — its tensor names contain `FakeQuantWithMinMaxVars` and `quant_` prefixes, which are quantization-aware training (QAT) artifacts. The BirdNET team retrained the model with fake quantization nodes. Standard PTQ applied to the SavedModel cannot replicate this.

## Next

16x8 PTQ — weights INT8, activations INT16 (`EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8`). Designed for audio/spectrogram models sensitive to INT8 activation quantization. Expected ~14 MB. Requires calibration.

In [6]:
import os
import sys
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

sys.path.insert(0, os.path.dirname(os.getcwd()))
import config

MODEL_PATH = "/Users/qian/Library/Application Support/birdnet/acoustic-models/v2.4/pb/model-fp32"
OUTPUT_PATH = os.path.join(config.OUTPUTS_DIR, "models", "birdnet_v2.4_int8_calibrated.tflite")
SAMPLE_RATE = 48000
N_SAMPLES = 144000   # 3s × 48kHz — matches SavedModel input shape (None, 144000)
N_CALIB = 100
RANDOM_SEED = 42

print("TF version:", tf.__version__)

TF version: 2.20.0


In [ ]:
# Sample 100 calibration clips from birdnet_species, stratified across recorders
labels = pd.read_csv(config.LABELS_PROGRESS_PATH)
species_clips = labels[labels["meaningful_source"] == "birdnet_species"].copy()

n_recorders = species_clips["Recorder"].nunique()
per_recorder = N_CALIB // n_recorders

calib_clips = (
    species_clips
    .groupby("Recorder", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), per_recorder), random_state=RANDOM_SEED))
    .sample(frac=1, random_state=RANDOM_SEED)
    .head(N_CALIB)
    .reset_index(drop=True)
)

print(f"Calibration clips: {len(calib_clips)}")
print(calib_clips["Recorder"].value_counts().to_string())

In [7]:
def load_audio(path):
    audio, sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo -> mono
    if len(audio) < N_SAMPLES:
        audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
    else:
        audio = audio[:N_SAMPLES]
    return audio.reshape(1, N_SAMPLES).astype(np.float32)


def representative_dataset():
    for _, row in calib_clips.iterrows():
        yield {"inputs": load_audio(row["audio_path"])}

In [2]:
import subprocess

SCRIPT_PATH = os.path.join(os.path.dirname(os.getcwd()), "scripts", "convert_birdnet_ptq.py")

print(f"Running: {SCRIPT_PATH}")
result = subprocess.run(
    [sys.executable, SCRIPT_PATH, OUTPUT_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:")
    print(result.stderr[-3000:])

Running: /Users/qian/KWF/rainforest-audio-detection/scripts/convert_birdnet_ptq.py
Converting...
FP32 TFLite:    51.7 MB
INT8 TFLite:    14.2 MB  (dynamic range)
Size reduction: 72.5%
Saved: /Users/qian/KWF/rainforest-audio-detection/outputs/models/birdnet_v2.4_int8.tflite



In [8]:
# Sanity check: run INT8 model on the same test clip used in notebook 07
# and compare top-1 prediction against the FP32 baseline
TEST_CLIP = "/Users/qian/KWF/Segmented_Foldered/Audio Moth 1/0_4999/Audio_Moth_1_20250317_093627.wav"

# Load labels (6522 species from the SavedModel)
from birdnet.acoustic_models.v2_4.pb import AcousticPBDownloaderV2_4
_, species_list = AcousticPBDownloaderV2_4.get_model_path_and_labels("en_us")
species_labels = list(species_list)

# Run INT8 TFLite
interpreter = tf.lite.Interpreter(model_path=OUTPUT_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

audio = load_audio(TEST_CLIP)
interpreter.set_tensor(inp["index"], audio)
interpreter.invoke()
scores = interpreter.get_tensor(out["index"])[0]

top5_idx = np.argsort(scores)[::-1][:5]
print("INT8 top-5 predictions:")
for i in top5_idx:
    print(f"  {species_labels[i]:<60} {scores[i]:.4f}")

# FP32 baseline from notebook 07: Spotted Antbird (0.3415), Checker-throated Stipplethroat (0.1126)
print("\nFP32 baseline top-1: Hylophylax naevioides_Spotted Antbird  0.3415")
print(f"INT8 top-1 match: {species_labels[top5_idx[0]]}")

INT8 top-5 predictions:
  Herpsilochmus dorsimaculatus_Spot-backed Antwren             -3.4067
  Ptilorrhoa caerulescens_Blue Jewel-babbler                   -3.5615
  Metopidius indicus_Bronze-winged Jacana                      -4.0261
  Eucometis penicillata_Gray-headed Tanager                    -4.0261
  Molothrus ater_Brown-headed Cowbird                          -4.1809

FP32 baseline top-1: Hylophylax naevioides_Spotted Antbird  0.3415
INT8 top-1 match: Herpsilochmus dorsimaculatus_Spot-backed Antwren


In [9]:
import time

FP32_TFLITE_PATH = "/Users/qian/Library/Application Support/birdnet/acoustic-models/v2.4/tf/model-fp32.tflite"
EVAL_N = 500

# Sample 500 clips from birdnet_species (fresh load, independent of b1000003)
labels_eval = pd.read_csv(config.LABELS_PROGRESS_PATH)
eval_clips = (
    labels_eval[labels_eval["meaningful_source"] == "birdnet_species"]
    .sample(EVAL_N, random_state=77)
    .reset_index(drop=True)
)
print(f"Evaluating {EVAL_N} clips...")

# Load both interpreters once
fp32_interp = tf.lite.Interpreter(model_path=FP32_TFLITE_PATH)
fp32_interp.allocate_tensors()
fp32_inp = fp32_interp.get_input_details()[0]
fp32_out = fp32_interp.get_output_details()[0]

int8_interp = tf.lite.Interpreter(model_path=OUTPUT_PATH)
int8_interp.allocate_tensors()
int8_inp = int8_interp.get_input_details()[0]
int8_out = int8_interp.get_output_details()[0]

def top1(interp, inp_d, out_d, audio):
    interp.set_tensor(inp_d["index"], audio)
    interp.invoke()
    return int(np.argmax(interp.get_tensor(out_d["index"])[0]))

agree = 0
t0 = time.time()
for i, (_, row) in enumerate(eval_clips.iterrows()):
    audio = load_audio(row["audio_path"])
    fp32_top1 = top1(fp32_interp, fp32_inp, fp32_out, audio)
    int8_top1 = top1(int8_interp, int8_inp, int8_out, audio)
    if fp32_top1 == int8_top1:
        agree += 1
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{EVAL_N}  agreement: {agree/(i+1)*100:.1f}%  ({time.time()-t0:.0f}s)")

print(f"\nTop-1 agreement: {agree}/{EVAL_N} = {agree/EVAL_N*100:.1f}%")

Evaluating 500 clips...


/Users/qian/miniforge3/envs/ds/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  100/500  agreement: 0.0%  (3s)
  200/500  agreement: 0.0%  (7s)
  300/500  agreement: 0.0%  (10s)
  400/500  agreement: 0.0%  (13s)
  500/500  agreement: 0.0%  (17s)

Top-1 agreement: 0/500 = 0.0%
